# CAMEL-AI LLM Generator - Phi-3-mini via Ollama

Genera conversaciones de CALIDAD REAL usando Phi-3-mini como backend LLM.
CAMEL-AI Inception Prompting: User Agent + Assistant Agent con roles.

**GPU**: Kaggle P100 (30h/sem)
**Modelo**: Phi-3-mini-4k-instruct (3.8B params)
**Output**:500K pares de alta calidad + JSONL + ShareGPT

In [ ]:
# ============================================================
# INSTALACION
# ============================================================
!pip install -q camel-ai aiohttp tqdm

# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

# Iniciar Ollama en background (compatible con Kaggle/Jupyter)
import subprocess, time, os
ollama_proc = subprocess.Popen(["ollama", "serve"], 
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)
time.sleep(5)
!ollama pull phi3:mini

# Verificar
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama list

In [ ]:
# ============================================================
# CAMEL-AI LLM GENERATOR
# ============================================================
import json, os, random, hashlib, asyncio, aiohttp
from pathlib import Path
from tqdm.notebook import tqdm

class CAMELAILLMGenerator:
    """
    CAMEL-AI Inception Prompting con Phi-3-mini:
    - User Agent: role-playea usuario con persona/topic especifico
    - Assistant Agent: role-playea experto en el tema
    - Genera conversaciones multi-turno REALES via LLM
    """
    
    def __init__(self):
        self.session = None
        self.conversations = []
        self.seen = set()
        
        # CAMEL-AI Roles para Inception Prompting
        self.user_roles = [
            ("estudiante", "Eres un estudiante universitario de 20 años. Hablas informal, usas jerga, a veces impreciso. Preguntas sobre programación, ciencia, vida."),
            ("profesional", "Eres un ingeniero senior de 35 años. Hablas técnico, preciso. Preguntas sobre arquitectura, devops, performance."),
            ("curioso", "Eres una persona curiosa de 28 años. Te fascina todo. Preguntas sobre ciencia, historia, filosofía."),
            ("frustrado", "Eres un usuario frustrado con problemas técnicos. Hablas directo. Reportas errores, bugs."),
            ("creativo", "Eres un escritor creativo de 30 años. Pides historias, poemas, ideas. Imaginativo."),
            ("docente", "Eres un profesor de 45 años. Explicas paso a paso, paciente. Pides explicaciones, ejemplos."),
            ("casual", "Eres una persona casual de 25 años. Hablas coloquial. Preguntas sobre vida, tecnología."),
            ("experto", "Eres un investigador de 40 años. Preguntas profundas, técnicas. State-of-the-art."),
        ]
        
        self.asst_roles = [
            ("profesor", "Eres un profesor universitario experto. Explicas con claridad, usas ejemplos, admites cuando no sabes."),
            ("ingeniero", "Eres un ingeniero senior. Respuestas técnicas, precisas, con código."),
            ("experto", "Eres un experto en múltiples campos. Respuestas profundas, con referencias."),
            ("amigo", "Eres un amigo cercano. Respuestas informales, empáticas, consejos prácticos."),
            ("tutor", "Eres un tutor paciente. Explicas paso a paso, con ejercicios."),
        ]
        
        self.topics = [
            "python", "javascript", "rust", "go", "react", "vue", "angular",
            "docker", "kubernetes", "aws", "azure", "linux", "git",
            "machine learning", "deep learning", "nlp", "computer vision",
            "matematicas", "fisica", "quimica", "biologia", "astronomia",
            "filosofia", "historia", "psicologia", "economia",
            "salud", "nutricion", "ejercicio", "meditacion",
            "cocina", "viajes", "musica", "arte", "fotografia",
            "emprendimiento", "marketing", "finanzas", "inversiones",
            "relaciones", "comunicacion", "liderazgo", "productividad",
        ]
    
    async def __aenter__(self):
        self.session = aiohttp.ClientSession(
            timeout=aiohttp.ClientTimeout(total=180)
        )
        return self
    
    async def __aexit__(self, *args):
        if self.session:
            await self.session.close()
    
    async def _ollama(self, prompt, temperature=0.7, num_predict=512):
        """Llama a Ollama API"""
        payload = {
            "model": "phi3:mini",
            "prompt": prompt,
            "temperature": temperature,
            "top_p": 0.9,
            "stream": False,
            "options": {"num_predict": num_predict}
        }
        try:
            async with self.session.post(
                "http://localhost:11434/api/generate",
                json=payload
            ) as resp:
                data = await resp.json()
                return data.get("response", "").strip()
        except Exception as e:
            return None
    
    async def generate_conversation(self):
        """Genera UNA conversacion multi-turno via CAMEL-AI LLM"""
        
        # Seleccionar roles y tema
        user_role = random.choice(self.user_roles)
        asst_role = random.choice(self.asst_roles)
        topic = random.choice(self.topics)
        
        # CAMEL-AI System Prompt (Inception Prompting)
        system = f"""<CAMEL-AI INCEPTION PROMPTING>
Eres un generador de conversaciones realistas en español.

CONTEXTO:
- User Agent: {user_role[1]}
- Assistant Agent: {asst_role[1]}
- Tema: {topic}

REGLAS:
1. Genera una conversación natural de 3-6 turnos
2. El User pregunta sobre {topic} de forma realista
3. El Assistant responde con información precisa y útil
4. Incluye follow-ups naturales del User
5. Español neutro, natural, variado
6. Respuestas de longitud variable (cortas y largas)
7. SIN formato markdown, solo texto plano

FORMATO DE SALIDA (JSON):
{{"turns": [{{"role": "user", "content": "..."}}, {{"role": "assistant", "content": "..."}}]}}
</CAMEL-AI>"""
        
        # Step 1: User genera primera pregunta
        user_prompt = f"Genera UNA pregunta natural y realista sobre {topic}. Solo la pregunta."
        first_msg = await self._ollama(
            f"{system}\n\n{user_prompt}",
            temperature=0.9,
            num_predict=100
        )
        
        if not first_msg or len(first_msg) < 5:
            return None
        
        turns = [{"role": "user", "content": first_msg}]
        
        # Step 2: Generar 2-5 turnos mas
        num_turns = random.randint(2, 5)
        
        for i in range(num_turns):
            # Assistant responde
            conv_context = "\n".join([
                f"{'Usuario' if t['role']=='user' else 'Asistente'}: {t['content']}"
                for t in turns
            ])
            
            asst_prompt = f"Conversación:\n{conv_context}\n\nContinúa como {asst_role[0]}. Responde de forma natural y útil."
            
            asst_resp = await self._ollama(
                f"{system}\n\n{asst_prompt}",
                temperature=0.7,
                num_predict=300
            )
            
            if not asst_resp or len(asst_resp) < 10:
                break
            
            turns.append({"role": "assistant", "content": asst_resp})
            
            # User follow-up (si no es ultimo)
            if i < num_turns - 1:
                followup_prompt = f"Conversación:\n{conv_context}\n\nAsistente: {asst_resp}\n\nContinúa como {user_role[0]}. Haz follow-up natural."
                
                fu = await self._ollama(
                    f"{system}\n\n{followup_prompt}",
                    temperature=0.9,
                    num_predict=100
                )
                
                if fu and len(fu) > 3:
                    turns.append({"role": "user", "content": fu})
        
        return turns
    
    def _is_quality(self, turns):
        """Filtra calidad"""
        if len(turns) < 4:
            return False
        for t in turns:
            if len(t["content"]) < 15:
                return False
        # Sin repeticiones
        texts = [t["content"] for t in turns]
        if len(set(texts)) != len(texts):
            return False
        return True
    
    async def generate_batch(self, size=8):
        """Genera batch de conversaciones"""
        tasks = [self.generate_conversation() for _ in range(size)]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        valid = []
        for r in results:
            if isinstance(r, list) and self._is_quality(r):
                h = hashlib.md5(json.dumps(r).encode()).hexdigest()[:16]
                if h not in self.seen:
                    self.seen.add(h)
                    valid.append(r)
        return valid
    
    def split_and_save(self, conversations, num_files=30):
        """Guarda en archivos"""
        output_dir = "/kaggle/working/soc_corpus"
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        
        per = len(conversations) // num_files
        rem = len(conversations) % num_files
        start = 0
        
        for i in range(num_files):
            end = start + per + (1 if i < rem else 0)
            batch = conversations[start:end]
            
            fn = f"chat_{6+i:02d}.txt"
            fp = os.path.join(output_dir, fn)
            
            with open(fp, "w", encoding="utf-8") as f:
                for conv in batch:
                    for t in conv:
                        r = "U" if t["role"]=="user" else "B"
                        f.write(f"{r}: {t['content']}\n")
                    f.write("\n")
            
            sz = os.path.getsize(fp)//1024
            print(f"  {fn}: {len(batch):,} ({sz} KB)")
            start = end
        
        # JSONL
        jl = os.path.join(output_dir, "soc_500k_llm.jsonl")
        with open(jl, "w", encoding="utf-8") as f:
            for c in conversations:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        
        # ShareGPT
        sg = os.path.join(output_dir, "soc_500k_llm_sharegpt.json")
        sg_data = [{"conversations": [{"from": t["role"], "value": t["content"]} for t in c]} for c in conversations]
        with open(sg, "w", encoding="utf-8") as f:
            json.dump(sg_data, f, ensure_ascii=False)
        
        return output_dir

print("CAMEL-AI LLM Generator loaded")

In [ ]:
# ============================================================
# EJECUCION PRINCIPAL
# ============================================================

async def main():
    async with CAMELAILLMGenerator() as gen:
        target = 500000
        pbar = tqdm(total=target, desc="CAMEL-AI LLM")
        
        while len(gen.conversations) < target:
            batch = await gen.generate_batch(8)
            
            if batch:
                gen.conversations.extend(batch)
                pbar.update(len(batch))
                
                # Checkpoint cada 5K
                if len(gen.conversations) % 5000 < 8:
                    print(f"\nCheckpoint: {len(gen.conversations):,}")
        
        # Save
        print(f"\nGuardando {len(gen.conversations):,} conversaciones...")
        out = gen.split_and_save(gen.conversations)
        
        print(f"\n{'='*60}")
        print(f"COMPLETADO: {len(gen.conversations):,} conversations LLM")
        print(f"Output: {out}")
        
        return gen.conversations

conversations = await main()

In [ ]:
# ============================================================
# VERIFICACION
# ============================================================
import os

output_dir = "/kaggle/working/soc_corpus"
files = sorted([f for f in os.listdir(output_dir) if f.startswith("chat_") and f.endswith(".txt")])
total = 0

for f in files:
    fp = os.path.join(output_dir, f)
    sz = os.path.getsize(fp)//1024
    with open(fp) as fh:
        pairs = fh.read().count("U: ")
    total += pairs
    print(f"{f}: {pairs:,} ({sz} KB)")

print(f"\nTotal: {total:,}")

In [ ]:
# ============================================================
# SUBIR A GITHUB
# ============================================================
!git config --global user.email "kaggle@rubidium.ai"
!git config --global user.name "Kaggle Bot"

repo = "/kaggle/working/rubidium-api"
if not os.path.exists(repo):
    !git clone https://github.com/diegovelandiabarajas1-lang/rubidium-api.git {repo}

res = f"{repo}/resources"
os.makedirs(res, exist_ok=True)

for f in files:
    os.system(f"cp {os.path.join(output_dir, f)} {os.path.join(res, f)}")

os.chdir(repo)
!git add resources/chat_*.txt
!git commit -m "feat: CAMEL-AI LLM 500K corpus - high quality" || true
!git push origin main || echo "Push failed"